# 🎙️ 무료 AI 목소리 들어 보기 (Qwen3-TTS)

알리바바가 공개한 무료 음성 모델(Apache-2.0, **상업적 사용 가능**)로 한국어 나레이션을 만들어 들어 봐요.
말로 목소리를 설명하면 그런 목소리를 새로 만들어 줘요.

### 사용법
1. **맨 먼저** 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장** (무료 GPU를 켜는 거예요)
2. **①** ▶ : 프로그램 설치 + 모델 받기 (처음 3~5분)
3. **②** ▶ : 목소리 3가지로 나레이션을 만들어서 바로 들어 볼 수 있어요. 문장과 목소리 설명은 바꿔서 다시 ▶ 해도 돼요.

In [ ]:
#@title ① 준비하기 (처음 3~5분) { display-mode: "form" }
import torch, subprocess, sys
if not torch.cuda.is_available():
    print('⚠️ GPU가 꺼져 있어요. 위 메뉴 런타임 → 런타임 유형 변경 → T4 GPU → 저장 후 다시 ▶ 눌러 주세요.')
else:
    print('GPU:', torch.cuda.get_device_name(0))
    print('프로그램 설치 중…')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'qwen-tts', 'soundfile'], capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr[-1500:])
    from qwen_tts import Qwen3TTSModel
    print('목소리 모델 받는 중… (약 4GB, 처음 한 번)')
    TTS = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign', device_map='cuda:0', dtype=torch.float16, attn_implementation='sdpa')
    print('✅ 준비 완료! 아래 ② 칸 ▶ 버튼을 누르세요.')


In [ ]:
#@title ② 목소리 만들어 들어 보기 { display-mode: "form" }
문장 = "엄마는 무너지는 마음을 붙잡고 기도를 시작합니다. 그런데 모두가 포기하려던 그때, 믿을 수 없는 일이 일어나는데요." #@param {type:"string"}
목소리1 = "A Korean male narrator in his 30s with a warm, slightly low voice, telling a movie story with emotion and suspense, natural Korean intonation, medium-fast pace" #@param {type:"string"}
목소리2 = "A calm, deep Korean male documentary narrator, sincere and moving, gentle emotional emphasis, natural Korean" #@param {type:"string"}
목소리3 = "A bright, friendly Korean female storyteller in her 20s, expressive and lively, like a popular YouTube movie recap narrator" #@param {type:"string"}
import soundfile as sf, time, os
from IPython.display import Audio, display
os.makedirs('/content/voice_test', exist_ok=True)
try:
    for i, desc in enumerate([목소리1, 목소리2, 목소리3], 1):
        if not desc.strip(): continue
        t0 = time.time()
        wavs, sr = TTS.generate_voice_design(text=문장, language='Korean', instruct=desc)
        path = f'/content/voice_test/voice{i}.wav'; sf.write(path, wavs[0], sr)
        print(f'🔊 목소리 {i}  ({time.time()-t0:.0f}초 걸림)  {desc[:60]}…')
        display(Audio(path))
    print('\n마음에 드는 번호와, 바꾸고 싶은 점(더 낮게, 더 빠르게, 더 감정적으로 등)을 알려 주세요.')
except NameError:
    print('먼저 위의 ① 칸 ▶ 버튼을 눌러 주세요.')
except Exception as e:
    print('문제가 생겼어요:', e)
